# API Valencia Air Pollution
This an excercise using the api in:
* https://opendata.vlci.valencia.es/dataset/estacions-contaminacio-atmosferiques-estaciones-contaminacion-atmosfericas
* https://opendata.vlci.valencia.es/

El proceso funciona algo diferente al video:
1. primero se require consultar los metadatos en CKAN para obtener la url de la API
2. segundo usar la url de la API

In [6]:
import sys
print(sys.executable)
print(sys.version)

e:\Advanced Python\.venv_valencia\Scripts\python.exe
3.14.9 (tags/v3.14.9:ead82c0, Oct 21 2026, 06:26:14) [MSC v.1944 64 bit (AMD64)]


In [14]:
import requests
import json
import pandas as pd

In [ ]:
def make_request(base_url, params):
  response = requests.get(base_url, params=params, timeout=30)
  if response.status_code == 200:
    return response.json()
  else:
    try:
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        print("Request failed:", e)
        raise

In [24]:
# 1. Consultar los metadatos del dataset en CKAN
ckan_url = "https://opendata.vlci.valencia.es/api/3/action/package_show"

# Se concatenan automáticamente strings cuando se usan paréntesis, sin necesidad de usar el operador +
dataset_id = (
    "estacions-contaminacio-atmosferiques-"
    "estaciones-contaminacion-atmosfericas"
)

response = make_request(ckan_url, params={"id": dataset_id})
print(json.dumps(response, indent=2))

{
  "help": "https://opendata.vlci.valencia.es/api/3/action/help_show?name=package_show",
  "success": true,
  "result": {
    "author": "",
    "author_email": "",
    "creator_user_id": "6caefb73-d160-4a2f-8e54-b9d3f76acce3",
    "data_source": "",
    "geographic_coverage": "",
    "id": "6d8c3706-f965-411b-aad8-e159f8ad8dcd",
    "isopen": true,
    "language": "",
    "license_id": "cc-by",
    "license_title": "Atribuci\u00f3n 4.0 Internacional (CC BY 4.0)",
    "license_url": "http://creativecommons.org/licenses/by/4.0/deed.es",
    "maintainer": "",
    "maintainer_email": "",
    "metadata_created": "2026-02-17T17:58:35.857962",
    "metadata_modified": "2026-10-14T11:02:24.854016",
    "name": "estacions-contaminacio-atmosferiques-estaciones-contaminacion-atmosfericas",
    "name_translated_en": "",
    "name_translated_es": "",
    "name_translated_va": "",
    "normatives": "",
    "notes": "Ubicaci\u00f3n de estaciones de contaminaci\u00f3n atmosf\u00e9ricas.\r\n\r\nFuente

In [17]:
# 2. Encontrar el recurso JSON de ArcGIS
resources = response["result"]["resources"]

# Buscar el primer recurso que sea de formato JSON y contenga "/query" en la URL
resource = next(
    (
        r for r in resources
        if r.get("format", "").upper() == "JSON"
        and "/query" in r.get("url", "")
    ),
    None,
)

if resource is None:
    raise RuntimeError("No se encontró el recurso ArcGIS JSON")

arcgis_url = resource["url"].split("?")[0]
print("ArcGIS URL:", arcgis_url)

ArcGIS URL: https://geoportal.valencia.es/server/rest/services/OPENDATA/MedioAmbiente/MapServer/156/query
